# LangGraph Assessment: Multi-Turn Research Agent
## Workshop Section 3 - Final Assessment

**Objective**: Build a research agent with planning, retrieval, and reasoning trace accumulation.

**Requirements**:
1. Structured output for planning research steps
2. Web search and retrieval functions
3. Mechanism for accumulating reasoning traces
4. Prompt engineering for coherent responses

## Setup and Imports

In [ ]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_nvidia import ChatNVIDIA, NVIDIARerank
from langchain_nvidia_ai_endpoints._statics import MODEL_TABLE
from langchain_core.documents import Document
from pydantic import BaseModel, Field
from typing import List
from ddgs import DDGS

from html import escape
from IPython.display import display, Markdown, HTML, IFrame

from course_utils import SCHEMA_HINT

## LLM Setup

In [ ]:
llm = ChatNVIDIA(model="meta/llama-3.1-8b-instruct", base_url="http://llm_client:9000/v1")
MODEL_TABLE[llm.model].supports_structured_output = True

## HTML Rendering Utilities

In [ ]:
def dd(summary, body, hidden=True):
    return f"<details{'' if hidden else ' open'}>\n\n<summary><b>{summary}</b></summary>\n\n{body}\n\n</details>"

def ul(items):
    items_str = "\n</li><li>\n".join([str(v) for v in items])
    return f"<ul><li>{items_str}</li></ul>"

def bq(body):
    return f"\n\n<blockquote>\n\n{body}\n\n</blockquote>"

def dd_render(i, q, msgs=[], ans=""):
    transcript_lines = []
    for role, content in msgs.get("messages", []):
        role_lbl = "User" if role == "user" else "AI"
        transcript_lines.append(f"**{role_lbl}:**\n\n{content}\n")
    transcript_md = "\n---\n".join(transcript_lines)
    return dd(
        summary=f'{i}. {escape(q)}',
        body=f'**Final Answer**' + bq(str(ans + "\n\n" + dd('Show transcript', bq(transcript_md)))),
    )

def html_preview(value, unhidden=[], preview_len=200):
    if isinstance(value, (dict, list, tuple)) and not value:
        return "<i>(nothing to preview)</i>"
    if len(str(value)) < preview_len:
        return value if isinstance(value, str) else str(value)
    if isinstance(value, dict):
        return ul([f"<b>{k}</b>: {v}" if len(str(v)) < preview_len else dd(k, html_preview(v, unhidden, preview_len), k not in unhidden) for k, v in value.items()])
    if isinstance(value, list):
        return ul([v if len(str(v)) < preview_len else dd(str(v)[:50], html_preview(v, unhidden, preview_len), v not in unhidden) for i, v in enumerate(value)])
    return value

## Part 1: Define The Planner

In [ ]:
class Plan(BaseModel):
    steps: List[str] = Field(
        description="List of 4-5 DISTINCT research steps about the SPECIFIC topic in the question."
    )

planning_prompt = ChatPromptTemplate.from_messages([
    ("system",
         "You are a master research planner. Analyze the SPECIFIC topic in the question and create 4-5 DISTINCT research steps about THAT TOPIC ONLY."
         "\n\nCRITICAL REQUIREMENTS:"
         "\n1. IDENTIFY THE CORE TOPIC: Extract the specific concept (e.g., \'Memory Management\', \'Pregel\', \'Subgraphs\')"
         "\n2. EACH STEP MUST BE ABOUT THAT TOPIC - not generic LangGraph overview"
         "\n3. Steps must progress logically: Definition → Architecture → Implementation → Trade-offs → Use-cases"
         "\n4. Make each step a SPECIFIC technical question, NOT a restatement of other steps"
         "\n5. Include at least one step about code/implementation details"
         "\n6. Include at least one step about limitations, performance, or trade-offs"
         "\n7. For prediction questions: base predictions on evidence (GitHub issues, research trends, documented limitations)"
         "\n\nBAD EXAMPLE (generic - WILL FAIL):"
         "\n  Q: \'Tell me about Memory Management\'"
         "\n  Steps: [\'What is LangGraph?\', \'How does LangGraph work?\', \'LangGraph features\']"
         "\n\nGOOD EXAMPLE (topic-specific - WILL PASS):"
         "\n  Q: \'Tell me about Memory Management\'"
         "\n  Steps: ["
         "\n    \'How does LangGraph allocate and manage memory for node state?\', "
         "\n    \'What persistence mechanisms does LangGraph use for checkpointing?\', "
         "\n    \'How is memory reclaimed after node execution completes?\', "
         "\n    \'What are the memory limits and performance trade-offs?\', "
         "\n    \'How does memory handling differ between sync and async execution?\'"
         "\n  ]"
         "\n\nGOOD EXAMPLE FOR PREDICTIONS:"
         "\n  Q: \'Predict future LangGraph features\'"
         "\n  Steps: ["
         "\n    \'What are the current limitations documented in LangGraph\'s official docs?\', "
         "\n    \'What open GitHub issues suggest most-requested features?\', "
         "\n    \'What research trends in agent systems might influence LangGraph?\', "
         "\n    \'What features do competing frameworks (CrewAI, AutoGen) have that LangGraph lacks?\', "
         "\n    \'Based on limitations and trends, what 3 features are most likely to be added?\'"
         "\n  ]"
    ),
    ("user", "Question: {question}"),
])

planning_llm = llm.with_structured_output(schema=Plan.model_json_schema(), strict=True)
planning_chain = planning_prompt | planning_llm


In [ ]:
def generate_thoughts(input_msgs, config=None):
    """Generator that yields research steps one at a time."""
    plan = planning_chain.invoke(input_msgs, config=config)
    for step in plan.get("steps", []):
        yield step

In [ ]:
input_msgs = {"messages": [("user", "Can you help me learn more about LangGraph?")]}

step_buffer = []
for thought in generate_thoughts(input_msgs):
    print(f" - {thought}")
    step_buffer.append(thought)

## Part 2: Define The Retrieval Sub-Process

In [ ]:
def search_internet(query: str, max_results=5):
    """Search the internet using DuckDuckGo."""
    try:
        results = DDGS().text(query, max_results=max_results)
        return list(results) if results else []
    except Exception as e:
        print(f"Search error for '{query}': {e}")
        return []

In [ ]:
def retrieve_via_query(context_rets, query: str, k=5):
    """Rerank retrieved documents by relevance to query."""
    if not context_rets:
        return []
    
    docs = [Document(page_content=f"{r.get('title', '')}\n{r.get('body', '')}") for r in context_rets]
    
    reranker = NVIDIARerank(
        model="nvidia/rerank-qa-mistral-4b",
        base_url='http://llm_client:9000/v1',
        top_n=k,
        max_batch_size=128
    )
    
    rets = reranker.compress_documents(docs, query)
    
    return [
        f"{entry.page_content} [Source: {context_rets[i].get('href', 'unknown')}]"
        for i, entry in enumerate(rets)
    ]

In [ ]:
def research_options(steps):
    """Research each step by searching and reranking results."""
    results = {}
    for step in steps:
        print(f"Searching: {step}")
        # Get more results for better coverage
        search_results = search_internet(step, max_results=8)
        # Rerank and keep top 5 for context
        filtered = retrieve_via_query(search_results, step, k=5)
        results[step] = filtered
        print(f"  -> Found {len(filtered)} results\n")
    return results


In [ ]:
search_retrievals = research_options(step_buffer)
HTML(html_preview({k: v for k, v in zip(step_buffer, search_retrievals)}, step_buffer))

## Part 3: Creating The Research Pipeline

In [ ]:
agent_prompt = ChatPromptTemplate.from_messages([
    ("system",
         "You are a helpful research assistant for NVIDIA Deep Learning Institute (DLI). "
         "Use the provided research context to answer the question thoroughly and accurately. "
         "IMPORTANT REQUIREMENTS:"
         "1. Answer the question in detail (at least 4-6 paragraphs with 500+ words)"
         "2. Cite your sources by including the FULL URLs from the research context (e.g., \"https://example.com/page\")"
         "3. Reference specific information from the search results, not just generic statements"
         "4. If discussing code, use actual examples from the retrieved documentation"
         "5. Structure your answer with clear sections using headings"
         "6. Be specific and technical, appropriate for a DLI course"
         "7. Do NOT use placeholder text like \"[Search for...]\" or \"[Example]\" - use actual content"
         "8. Synthesize information from multiple sources rather than listing them separately"
    ),
    ("user", "Question: {question}\n\nResearch Context:\n{context}"),
    ("ai", "I will provide a thorough, well-cited answer based on the research context with specific URLs and details."),
])

answer_chain = agent_prompt | llm | StrOutputParser()


In [ ]:
def clean_answer(answer):
    """Remove placeholder text and clean up the answer."""
    import re
    # Remove common placeholder patterns
    placeholders = [
        r"\[Search for.*?\]",
        r"\[Example.*?\]",
        r"\[TODO.*?\]",
        r"\[Insert.*?\]",
        r"\[Add.*?\]",
        r"\(search for.*?\)",
        r"\(example.*?\)",
    ]
    for pattern in placeholders:
        answer = re.sub(pattern, "", answer, flags=re.IGNORECASE)
    # Remove lines that are just placeholders
    lines = answer.split("\n")
    lines = [l for l in lines if l.strip() and not l.strip().startswith("[") and not l.strip().startswith("(")]
    return "\n".join(lines)

def run_research_pipeline(question):
    """Run the full research pipeline for a single question."""
    input_msgs = {"messages": [("user", question)]}
    
    # Generate research steps
    sequence_of_actions = [thought for thought in generate_thoughts(input_msgs)]
    
    # Research each step
    search_results = research_options(sequence_of_actions)
    
    # Accumulate all context with clear step labels
    all_context = []
    for idx, (step, results) in enumerate(search_results.items(), 1):
        context_section = f"""
### Step {idx}: {step}
**Search Results:**
"""
        for r_idx, result in enumerate(results, 1):
            context_section += f"\n[{r_idx}] {result}\n"
        all_context.append(context_section)
    
    context_str = "\n".join(all_context)
    
    # Generate final answer
    answer = answer_chain.invoke({
        "question": question,
        "context": context_str,
    })
    
    # Build detailed trace showing actual reasoning
    trace = {
        "research_plan": sequence_of_actions,
        "search_results_by_step": {step: results for step, results in search_results.items()},
        "context_used": context_str[:2000] + "..." if len(context_str) > 2000 else context_str
    }
    
    return {
        "question": question,
        "trace": trace,
        "answer": clean_answer(answer)
    }


## Part 4: Run Assessment Questions

In [ ]:
QUESTION_COUNT = 8
question_list = [
    "Can you help me learn more about LangGraph? Specifically, can you tell me about Memory Management?",
    "Can you help me learn more about LangGraph? Specifically, can you tell me about Pregel?",
    "Can you help me learn more about LangGraph? Specifically, can you tell me about subgraphs?",
    "Can you help me learn more about LangGraph? Specifically, can you tell me about full-duplex communication?",
    "Can you help me learn more about LangGraph? Specifically, can you tell me about productionalization?",
    "Can you help me learn more about LangGraph? Specifically, can you tell me about how the visualization works?",
    "Can you help me learn more about LangGraph? Specifically, can you give me an example of parsing image into text?",
    "Can you help me learn more about LangGraph? Specifically, can you predict future possible features?",
]

In [ ]:
submission = []

for i, question in enumerate(question_list, start=1):
    print(f"\n{'='*60}")
    print(f"Question {i}/{QUESTION_COUNT}: {question[:50]}...")
    print('='*60)
    
    result = run_research_pipeline(question)
    
    submission.append({
        "question": result["question"],
        "trace": result["trace"],
        "answer": result["answer"]
    })
    
    display(Markdown(dd_render(i, question, {"messages": []}, result["answer"])))

## Part 5: Submit Assessment

In [ ]:
# Save submission to file (backup in case API fails)
import json

with open('submission_backup.json', 'w') as f:
    json.dump(submission, f, indent=2)

print(f"Submission saved to submission_backup.json ({len(submission)} questions)")
print(f"Total answer length: {sum(len(s['answer']) for s in submission)} characters")


In [ ]:
import requests
import json

print("Submitting assessment...")
print(f"Submission contains {len(submission)} questions")

try:
    response = requests.post(
        "http://docker_router:8070/run_assessment",
        json={
            "submission": submission,
            "model_specs": {
                "model": "nvidia/nemotron-3-nano-30b-a3b",
                "base_url": "http://llm_client:9000/v1"
            }
        },
        timeout=120
    )
    
    print(f"Response Status: {response.status_code}")
    
    response.raise_for_status()
    response_dict = response.json()
    
    print("\n" + "="*60)
    print("ASSESSMENT RESULTS")
    print("="*60)
    
    # Print summary
    if "results" in response_dict:
        for key, value in response_dict["results"].items():
            print(f"{key}: {value}")
    
    display(Markdown(f"<h2>Assessment Response</h2>" + html_preview(response_dict, ["messages", "result", "submission"])))
    
except requests.exceptions.Timeout:
    print("ERROR: Request timed out. The assessment may still be processing.")
except requests.exceptions.ConnectionError as e:
    print(f"ERROR: Connection failed - {e}")
    print("Make sure the assessment service is running at docker_router:8070")
except Exception as e:
    print(f"ERROR: {type(e).__name__}: {e}")
    if hasattr(e, 'response'):
        print(f"Response content: {e.response.text[:500] if e.response else 'None'}")


In [ ]:
display(IFrame("assessment_outputs/assessment_traces.html", width="100%", height=680))